# Copy missing onset `.mat` files into raw fMRI sessions

This notebook scans every `sub-*/ses-*` folder under the raw fMRI root. If a session does not already contain any `.mat` files, it searches the onset folder for files whose names contain the same participant ID and session label, then copies the matches into that session folder.

Update the two path variables in the next cell if your directories change.

In [ ]:
from pathlib import Path
import shutil

RAW_ROOT = Path(r"E:\03-Raw_fMRI")
ONSET_ROOT = Path(r"E:\03-Raw_fMRI\onsets\dr17_20241114f_tasks_events\task-memory_events")

assert RAW_ROOT.exists(), f"Raw fMRI root not found: {RAW_ROOT}"
assert ONSET_ROOT.exists(), f"Onset root not found: {ONSET_ROOT}"

: 

In [ ]:
def subject_and_session(session_dir: Path) -> tuple[str, str]:
    participant = session_dir.parent.name
    session = session_dir.name
    if not participant.startswith("sub-") or not session.startswith("ses-"):
        raise ValueError(f"Unexpected session path: {session_dir}")
    return participant.removeprefix("sub-"), session.removeprefix("ses-")

def matching_onset_files(participant: str, session: str) -> list[Path]:
    participant = participant.lower()
    session = session.lower()
    matches = []
    for mat_path in ONSET_ROOT.rglob("*.mat"):
        name = mat_path.name.lower()
        if participant in name and session in name:
            matches.append(mat_path)
    return sorted(matches)

def copy_missing_matfiles(raw_root: Path = RAW_ROOT, onset_root: Path = ONSET_ROOT, dry_run: bool = True) -> list[dict[str, str]]:
    results: list[dict[str, str]] = []
    for session_dir in sorted(raw_root.glob("sub-*/ses-*")):
        if not session_dir.is_dir():
            continue

        participant, session = subject_and_session(session_dir)
        local_mats = list(session_dir.rglob("*.mat"))

        if local_mats:
            results.append({
                "session": str(session_dir),
                "status": "skipped_existing",
                "count": str(len(local_mats)),
            })
            continue

        matches = matching_onset_files(participant, session)
        if not matches:
            results.append({
                "session": str(session_dir),
                "status": "no_match",
                "count": "0",
            })
            continue

        copied = 0
        for source_path in matches:
            destination_path = session_dir / source_path.name
            if destination_path.exists():
                continue
            if not dry_run:
                shutil.copy2(source_path, destination_path)
            copied += 1

        results.append({
            "session": str(session_dir),
            "status": "copied" if copied else "matched_but_already_present",
            "count": str(copied),
        })

    return results

In [ ]:
# Dry run first so you can review what would be copied.
report = copy_missing_matfiles(dry_run=True)
for row in report:
    print(row)

print(f"Checked {len(report)} session folders.")

## To perform the copy

After reviewing the dry-run output, rerun the same function with `dry_run=False`.

In [ ]:
# Uncomment to actually copy files.
# report = copy_missing_matfiles(dry_run=False)
# for row in report:
#     print(row)